# Step 5 — Explore the raw ticket data
Support Ticket Insights Agent — Phase 2, Step 5.

Goal: look at columns/types, missing values, duplicates, value distributions, and a handful of raw rows read by eye, before designing the clean data model (Step 7).

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
df = pd.read_csv('../data/raw/customer_support_tickets.csv')
df.shape

(8469, 17)

## Columns and dtypes

In [2]:
df.dtypes

Ticket ID                         int64
Customer Name                       str
Customer Email                      str
Customer Age                      int64
Customer Gender                     str
Product Purchased                   str
Date of Purchase                    str
Ticket Type                         str
Ticket Subject                      str
Ticket Description                  str
Ticket Status                       str
Resolution                          str
Ticket Priority                     str
Ticket Channel                      str
First Response Time                 str
Time to Resolution                  str
Customer Satisfaction Rating    float64
dtype: object

## Missing values per column

In [3]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

,missing_count,missing_pct
Customer Satisfaction Rating,5700,67.3
Time to Resolution,5700,67.3
Resolution,5700,67.3
First Response Time,2819,33.3
Ticket Description,0,0.0
Ticket Channel,0,0.0
Ticket Priority,0,0.0
Ticket Status,0,0.0
Ticket ID,0,0.0
Customer Name,0,0.0


## Blank strings (not NaN, but empty) — CSVs often hide these
Some columns showed as non-null in a quick head() but were empty strings, e.g. `Resolution`.

In [4]:
blank = (df.astype(str).apply(lambda s: s.str.strip()) == '').sum().sort_values(ascending=False)
blank[blank > 0]

Series([], dtype: int64)

## Duplicates

In [5]:
print('duplicate Ticket ID rows:', df['Ticket ID'].duplicated().sum())
print('fully duplicate rows:', df.duplicated().sum())
print('duplicate (Customer Email, Ticket Subject, Ticket Description) rows:',
      df.duplicated(subset=['Customer Email', 'Ticket Subject', 'Ticket Description']).sum())

duplicate Ticket ID rows: 0
fully duplicate rows: 0
duplicate (Customer Email, Ticket Subject, Ticket Description) rows: 0


## Categorical value distributions

In [6]:
for col in ['Ticket Type', 'Ticket Status', 'Ticket Priority', 'Ticket Channel', 'Customer Gender']:
    print(f'--- {col} ---')
    print(df[col].value_counts(dropna=False))
    print()

--- Ticket Type ---
Ticket Type
Refund request          1752
Technical issue         1747
Cancellation request    1695
Product inquiry         1641
Billing inquiry         1634
Name: count, dtype: int64

--- Ticket Status ---
Ticket Status
Pending Customer Response    2881
Open                         2819
Closed                       2769
Name: count, dtype: int64

--- Ticket Priority ---
Ticket Priority
Medium      2192
Critical    2129
High        2085
Low         2063
Name: count, dtype: int64

--- Ticket Channel ---
Ticket Channel
Email           2143
Phone           2132
Social media    2121
Chat            2073
Name: count, dtype: int64

--- Customer Gender ---
Customer Gender
Male      2896
Female    2887
Other     2686
Name: count, dtype: int64



## Product Purchased — how many distinct products, and top counts

In [7]:
print('distinct products:', df['Product Purchased'].nunique())
df['Product Purchased'].value_counts().head(15)

distinct products: 42


Product Purchased
Canon EOS              240
GoPro Hero             228
Nest Thermostat        225
Philips Hue Lights     221
Amazon Echo            221
LG Smart TV            219
Sony Xperia            217
Roomba Robot Vacuum    216
LG OLED                213
Apple AirPods          213
iPhone                 212
Sony 4K HDR TV         210
LG Washing Machine     208
Garmin Forerunner      208
Canon DSLR Camera      206
Name: count, dtype: int64

## Date fields — ranges and formats

In [8]:
for col in ['Date of Purchase', 'First Response Time', 'Time to Resolution']:
    parsed = pd.to_datetime(df[col], errors='coerce')
    print(f'--- {col} ---')
    print('non-null:', parsed.notna().sum(), '/', len(df))
    print('min:', parsed.min(), ' max:', parsed.max())
    print()

--- Date of Purchase ---
non-null: 8469 / 8469
min: 2020-01-01 00:00:00  max: 2021-12-30 00:00:00

--- First Response Time ---
non-null: 5650 / 8469
min: 2023-05-31 21:55:39  max: 2023-06-02 00:54:21

--- Time to Resolution ---
non-null: 2769 / 8469
min: 2023-05-31 21:53:30  max: 2023-06-02 00:55:33



**Note:** `First Response Time` and `Time to Resolution` should be checked for whether they cluster on a single day (a red flag that they're generation timestamps, not real activity times).

In [9]:
resp = pd.to_datetime(df['First Response Time'], errors='coerce')
print('distinct calendar dates in First Response Time:', resp.dt.date.nunique())
res = pd.to_datetime(df['Time to Resolution'], errors='coerce')
print('distinct calendar dates in Time to Resolution:', res.dt.date.nunique())

distinct calendar dates in First Response Time: 3
distinct calendar dates in Time to Resolution: 3


## Customer Satisfaction Rating — only present for closed tickets?

In [10]:
pd.crosstab(df['Ticket Status'], df['Customer Satisfaction Rating'].isna(), rownames=['Status'], colnames=['Rating is NaN'])

Rating is NaN,False,True
Status,,
Closed,2769,0
Open,0,2819
Pending Customer Response,0,2881


## Ticket Description — check for the templated `{product_purchased}` placeholder

In [11]:
has_placeholder = df['Ticket Description'].str.contains(r'\{product_purchased\}', regex=True, na=False)
print('rows with literal {product_purchased} placeholder:', has_placeholder.sum(), '/', len(df))
print('pct:', round(has_placeholder.mean() * 100, 1), '%')

rows with literal {product_purchased} placeholder: 8469 / 8469
pct: 100.0 %


## Resolution text — spot check for templated/gibberish text

In [12]:
df.loc[df['Resolution'].notna() & (df['Resolution'].str.strip() != ''), 'Resolution'].sample(8, random_state=0).tolist()

['Five tree those particular they product officer various.',
 'School fight role shake story low inside next.',
 'Little perform detail player boy same study him.',
 'Grow laugh little air rest successful.',
 'Represent either radio quickly piece leave writer now.',
 'Product thought own walk six money.',
 'Fill put mind collection reality mission professor.',
 'Play trip employee guess.']

## Raw rows, read by eye

In [13]:
with pd.option_context('display.max_colwidth', 200):
    display(df.sample(5, random_state=1)[['Ticket ID', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution']])

,Ticket ID,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution
1825,1826,Billing inquiry,Hardware issue,"I'm having an issue with the {product_purchased}. Please assist. Thanks in return.""\n\n""Thanks for understanding.""\n\n""Thank you for sharing.""\n\n""Sorry.""\n\n""Please do not forget. Sorry I've chec...",Closed,Any because size far record.
473,474,Technical issue,Hardware issue,"I'm having an issue with the {product_purchased}. Please assist. */ protected <StringProperty Name = ""product_purchased_ID"" Value = ""5C0F4C7E2-845F- I've performed a factory reset on my {product_p...",Open,NaN
3509,3510,Refund request,Product recommendation,"I'm encountering a software bug in the {product_purchased}. Whenever I try to perform a specific action, the application crashes. Are there any updates or fixes available?\n\nIt is possible to set...",Open,NaN
5415,5416,Refund request,Software bug,I'm having an issue with the {product_purchased}. Please assist.\n\nPlease help with the order fulfillment process.\n\nThank you for your patience and patience with our delivery as we worked with ...,Pending Customer Response,NaN
5537,5538,Technical issue,Software bug,I'm having an issue with the {product_purchased}. Please assist.\n\n[16:43:25]Gnome: * [M.M.T.] the addressing service has found the device. The {product_ I've noticed a peculiar error message pop...,Closed,Debate wife window kitchen inside tonight give gas.
